# NB02 · 第一次训练闭环

| | |
|---|---|
| **目标** | 亲手把一个 policy 从数据训到能跑，并得到第一个 success rate——从此你评价任何模型都有了亲手的参照系 |
| **前置** | NB01 完成；GPU（本地或云） |
| **预计耗时** | 半天人工 + 2–6h 机器 |
| **产出物** | checkpoint + `results/NB02.json`（success rate 等） |
| **通过标准** | success rate 达到官方参考值 ±10%，或有完整的排查记录 |

规则：从上到下顺序执行；每个 ✅ 检查点必须核对；最后的复盘必须填写并 commit。


In [ ]:
from pathlib import Path
import subprocess, sys
import nbutils

CONFIG = {
    "policy": "diffusion",          # 或 "act"
    "env": "pusht",
    "repo_id": "lerobot/pusht",
    "sanity_steps": 2000,           # 先证明管道通
    "full_steps": 100_000,          # 正式训练（云上跑）
    "out": Path("outputs/nb02"),
}
CONFIG["out"].mkdir(parents=True, exist_ok=True)

In [ ]:
# 你安装的 LeRobot 版本的训练入口和参数（API 随版本变化，以这个输出为准）
print(subprocess.run([sys.executable, "-m", "lerobot.scripts.train", "--help"],
                     capture_output=True, text=True).stdout[:3000])

### 命令模板（按上面 --help 的实际参数名微调）

```bash
# sanity（笔记本里直接跑，~10 分钟）：只验证 数据→模型→反向传播→存 checkpoint 全通
python -m lerobot.scripts.train \
  --policy.type=diffusion --env.type=pusht \
  --dataset.repo_id=lerobot/pusht \
  --steps=2000 --output_dir=outputs/nb02/sanity

# full（放 tmux 或云上跑，别占着 notebook）
python -m lerobot.scripts.train \
  --policy.type=diffusion --env.type=pusht \
  --dataset.repo_id=lerobot/pusht \
  --steps=100000 --output_dir=outputs/nb02/full
```

**纪律**：sanity 没绿之前不许提交 full——浪费的 GPU 时都是白花的钱。


In [ ]:
# sanity run（阻塞约 10 分钟；失败就把 stderr 贴出来逐行读——读错误信息是核心技能）
cmd = [sys.executable, "-m", "lerobot.scripts.train",
       f"--policy.type={CONFIG['policy']}", f"--env.type={CONFIG['env']}",
       f"--dataset.repo_id={CONFIG['repo_id']}",
       f"--steps={CONFIG['sanity_steps']}", f"--output_dir={CONFIG['out']}/sanity"]
print(" ".join(cmd))
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:]); print(r.stderr[-2000:])
# ✅ 检查点：loss 在下降吗？checkpoint 文件存在吗？

In [ ]:
# 评测：优先用官方 eval 脚本（它处理了 obs 预处理/归一化的全部细节）
CKPT = str(CONFIG["out"] / "full")   # full 训完后改成实际 checkpoint 路径
N_EVAL = 50

print(subprocess.run([sys.executable, "-m", "lerobot.scripts.eval", "--help"],
                     capture_output=True, text=True).stdout[:1500])
# 然后照 --help 组装，例如：
# python -m lerobot.scripts.eval --policy.path=<CKPT> --env.type=pusht --eval.n_episodes=50 --eval.batch_size=10

In [ ]:
# 结果登记：eval 脚本会输出 success rate（stdout 或 eval_info.json）。
# 若脚本输出格式和预期不同，人肉填进 MANUAL 里——数字必须落盘，这是账本。
MANUAL = {
    "success_rate": None,     # 例如 0.62
    "n_eval": N_EVAL,
    "train_steps": CONFIG["full_steps"],
    "train_minutes": None,
    "official_reference": None,   # 官方 README/模型卡里的数字，写明出处
}
assert MANUAL["success_rate"] is not None, "填入你的实测 success rate 再继续"
lo, hi = nbutils.wilson_ci(int(MANUAL["success_rate"] * MANUAL["n_eval"]), MANUAL["n_eval"])
MANUAL["ci95"] = [round(lo, 3), round(hi, 3)]
print(f"success = {MANUAL['success_rate']:.1%}  95% CI [{lo:.1%}, {hi:.1%}]")
nbutils.log_result("NB02", MANUAL)

## 分析

1. **对标**：你的数字落在官方参考值 ±10% 内吗？注意先看 CI——n=50 时 CI 宽度约 ±13 个百分点，"差 5 个点"很可能只是噪声（NB03 会把这件事变成定量直觉）。
2. **若显著偏低，按顺序排查**（每查一项记一行笔记）：
   - 评测协议：n_episodes、初始状态分布、max steps 和官方一致吗？
   - 数据：用的数据集版本、episode 数一致吗？
   - 训练：steps 够吗？loss 收敛了吗？学习率是默认值吗？
   - 环境：渲染后端、随机 seed？
3. **成本账**：这次训练花了多少 GPU 分钟 / 多少钱？除以 success rate 的提升，这是你第一个「数据/算力 → 性能」的换算率。


## 复盘（必填，不填不算完成这本 notebook）

> 复盘写在这里并 commit。允许粗糙，禁止事后美化。

- **预期 vs 实际**：
- **最大的一个意外**：
- **卡最久的一步和根因**：
- **用一句话向非技术人解释本次学到的东西**：
- **进入下一本之前要做的一个动作**：
